# 1. Initializing BiJuTy

In [ ]:
import bijuty

# 2. Running PySpark Example

## 2.1 Downloading the data

In [ ]:
import os
import requests

# Create output directory
os.makedirs("uber_data", exist_ok=True)

# Data URL
repo_url = "https://raw.githubusercontent.com/fivethirtyeight/uber-tlc-foil-response/master/uber-trip-data/"

# All CSV files in the repository
csv_files = [
    "uber-raw-data-apr14.csv",
    "uber-raw-data-may14.csv",
    "uber-raw-data-jun14.csv",
    "uber-raw-data-jul14.csv",
    "uber-raw-data-aug14.csv",
    "uber-raw-data-sep14.csv",
]

print("Downloading Uber NYC trip data...\n")

# Download each file
for filename in csv_files:
    url = f"{repo_url}{filename}"
    output_path = os.path.join("uber_data", filename)
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Check for errors
        
        with open(output_path, 'wb') as f:
            f.write(response.content)
        
        file_size = len(response.content) / 1024 / 1024  # Convert to MB
        print(f"✓ {filename:<30} ({file_size:.2f} MB)")
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Failed to download {filename}: {e}")

print(f"\nDownloaded all files to: {os.path.abspath('uber_data')}")
print(f"Total files: {len([f for f in os.listdir('uber_data') if f.endswith('.csv')])}")

## 2.2 Processing the data using PySpark

In [ ]:
import time
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

def build_spark():
    cores = os.environ.get("SPARK_EXECUTOR_CORES")
    return (
        SparkSession.builder
        .appName("Parallelism_Demo")
        .master(f"spark://{os.environ.get('SPARK_MASTER_HOST')}:{os.environ.get('SPARK_MASTER_PORT')}")
        .config("spark.executor.instances", "1")
        .config("spark.executor.cores", cores)
        .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY"))
        .config("spark.sql.shuffle.partitions", cores)
        .getOrCreate()
    )

def execute_workload():
    # Load and format data
    folder_path = "uber_data/"
    uber_df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(folder_path)
    
    uber_df = uber_df.repartition(int(os.environ.get("SPARK_EXECUTOR_CORES")))
    
    uber_df = uber_df.withColumn("Date/Time", F.to_timestamp(F.col("Date/Time"), "M/d/yyyy H:mm:ss")) \
                 .withColumn("Date", F.to_date(F.col("Date/Time"), "M/d/yyyy H:mm:ss")) \
                 .withColumn("Time", F.date_format(F.col("Date/Time"), "HH:mm:ss")) \
                 .withColumn("pickup_hour", F.hour(F.col("Time")))

    print("Total rows loaded: ", uber_df.count())
    
    # Trips per day
    uber_df.groupBy("Date").agg(F.count("*").alias("total_trips")).orderBy(F.col("total_trips").desc()).collect()

    # Trips per Base
    uber_df.groupBy("Base").agg(F.count("*").alias("total_trips")).orderBy("total_trips", ascending=False).collect()
    
    # Peak hours
    uber_df.groupBy("pickup_hour").agg(F.count("*").alias("trips")).orderBy("pickup_hour").collect()
    


In [ ]:
# Initialize Spark context
spark = build_spark()

# Execute the workload
execute_workload()

In [ ]:
# Stop the spark context
spark.stop()

# Stop the cluster from the GUI